 # 03a_RLVR_Auto_Analysis_v24.ipynb



 ### 🤖 Automated Walk-Forward Blotter Ingestion & Diversification Audit (Gen 18)

 - **Track 1: Continuous Walk-Forward Leaderboard**: Scans and parses authoritative Parquet blotters.

 - **Gen 18 Ex-Post Capital Ensembling Verification**: Enforces linear return combination $r_{\text{blend}, t} = \frac{1}{M}\sum r_{m, t}$.

 - **Covariance & Diversification Ratio ($DR$) Decomposition**: Mathematically proves portfolio variance reduction.

 - **3-Row Synchronized Performance Architecture**: Cumulative Equity, Strict Geometric Alpha Ratio ($V_p / V_{\text{bm}}$), and Underwater Drawdown.

In [1]:
# CELL 1: Setup & Environment Initialization
import gc
import os
import pickle
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch

from core.paths import LOCAL_DATA_DIR, OUTPUT_DIR
from core.quant import QuantUtils
from core.settings import CacheConfig, TradingConfig
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from rl_discovery.adapter import RLVRGymEnv, make_eval_env
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.validator import AgentEvaluator

# Universal Path Resolution
RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive  # type: ignore[import-untyped]

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3"
    )
else:
    rl_root_default = Path(
        r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3"
    )
    rl_root = rl_root_default if rl_root_default.exists() else Path.cwd()

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

checkpoint_dir = OUTPUT_DIR / "model_checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(
    f"🚀 Evaluation Engine Initialized | Platform: {'Google Colab' if RUNNING_IN_COLAB else 'Local Workstation'} | Device: {device}"
)
print(f"📁 Checkpoint Root: {checkpoint_dir}")
print(f"📁 Output Destination: {OUTPUT_DIR}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades

🚀 Evaluation Engine Initialized | Platform: Local Workstation | Device: cpu
📁 Checkpoint Root: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output\model_checkpoints
📁 Output Destination: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output


In [2]:
# CELL 2: Pre-load Market Data & Precompute Base Returns
print("\n" + "=" * 65)
print("📦 PRE-LOADING MARKET DATA & ALPHACACHE...")
print("=" * 65)

base_config = TradingConfig()
benchmark_sym = base_config.benchmark

data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df

df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, base_config.calendar_ticker)

cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
feature_cube = pd.read_parquet(cache_file_path)

reward_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
reward_matrix["CASH"] = 0.0

valid_dates = set(feature_cube.index.get_level_values("Date").unique())
valid_calendar = trading_calendar[trading_calendar.isin(valid_dates)]

print(
    f"📅 Valid Master Calendar: {valid_calendar.min().strftime('%Y-%m-%d')} to {valid_calendar.max().strftime('%Y-%m-%d')} ({len(valid_calendar)} sessions)"
)
print(
    f"🏛️ Dynamic Benchmark: {benchmark_sym} | Default Holding Period: {base_config.holding_period} sessions"
)
print("=" * 65)

del df_ohlcv
gc.collect()


📦 PRE-LOADING MARKET DATA & ALPHACACHE...
📅 Valid Master Calendar: 1998-01-02 to 2026-09-25 (7227 sessions)
🏛️ Dynamic Benchmark: SPY | Default Holding Period: 5 sessions


20

In [3]:
# ### CELL 3: Track 1 — Ingest & Benchmark Continuous Walk-Forward Blotters

# %%
import re
from typing import Any, Dict, Tuple
import numpy as np
import pandas as pd
from core.paths import OUTPUT_DIR


def calculate_blotter_metrics(df_blotter: pd.DataFrame) -> Dict[str, Any]:
    """Computes standardized portfolio performance metrics from an authoritative blotter."""
    b_df = df_blotter.copy()
    date_col = b_df["date"] if "date" in b_df.columns else b_df["decision_date"]
    b_df["date"] = pd.to_datetime(date_col)
    b_df = b_df.sort_values("date").reset_index(drop=True)

    daily_net = b_df["net_daily_simple_ret"].to_numpy(dtype=float)
    daily_bm = b_df["bm_daily_simple_ret"].to_numpy(dtype=float)
    daily_spread = daily_net - daily_bm

    # Strict Geometric Compounding Invariant
    cum_agent = float(np.prod(1.0 + daily_net) - 1.0)
    cum_bm = float(np.prod(1.0 + daily_bm) - 1.0)
    excess_ret = float(cum_agent - cum_bm)

    mean_ret = float(np.mean(daily_net))
    std_ret = float(np.std(daily_net, ddof=1)) if len(daily_net) > 1 else 1e-8
    sharpe = float((mean_ret / std_ret) * np.sqrt(252)) if std_ret > 1e-8 else 0.0

    mean_sp = float(np.mean(daily_spread))
    std_sp = float(np.std(daily_spread, ddof=1)) if len(daily_spread) > 1 else 1e-8
    ir = float((mean_sp / std_sp) * np.sqrt(252)) if std_sp > 1e-8 else 0.0

    downside = daily_spread[daily_spread < 0]
    downside_std = float(np.std(downside, ddof=1)) if len(downside) > 1 else 1e-8
    sortino = (
        float((mean_sp / downside_std) * np.sqrt(252)) if downside_std > 1e-8 else 0.0
    )

    eq = b_df["agent_equity"].to_numpy(dtype=float)
    peaks = np.maximum.accumulate(eq)
    dds = (eq - peaks) / np.maximum(peaks, 1e-8)
    max_dd = float(np.min(dds))

    bm_var = float(np.var(daily_bm, ddof=1)) if len(daily_bm) > 1 else 1e-8
    beta = float(np.cov(daily_net, daily_bm)[0, 1] / bm_var) if bm_var > 1e-8 else 1.0

    return {
        "Total Return": cum_agent,
        "BM Return": cum_bm,
        "Excess Return": excess_ret,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "IR": ir,
        "Max DD": max_dd,
        "Beta": beta,
        "Sessions": len(daily_net),
        "Start Date": pd.to_datetime(b_df["date"].iloc[0]).strftime("%Y-%m-%d"),
        "End Date": pd.to_datetime(b_df["date"].iloc[-1]).strftime("%Y-%m-%d"),
    }


def parse_blotter_sort_key(filename: str) -> Tuple[int, int, int, str]:
    """
    Deterministic hierarchical sort key:
      1. Generation number (ascending integer, e.g. gen17 -> 17)
      2. Tier rank:
         - 0: ex_post_blend (always lead constituent of the generation)
         - 1: constituent seeds (sorted numerically by seed ID)
         - 2: committee or other combination blotters
         - 3: unclassified / fallback
      3. Seed ID (numerical: 42, 101, 777, 1337, 2026)
      4. Canonical filename (tie-breaker)
    """
    gen_match = re.search(r"gen(\d+)", filename, re.IGNORECASE)
    gen_num = int(gen_match.group(1)) if gen_match else 9999

    fname_lower = filename.lower()
    if "ex_post_blend" in fname_lower:
        tier_rank = 0
        seed_num = 0
    else:
        seed_match = re.search(r"_s(\d+)(?:_|\.|$)", fname_lower)
        if seed_match:
            tier_rank = 1
            seed_num = int(seed_match.group(1))
        elif "committee" in fname_lower:
            tier_rank = 2
            seed_num = 0
        else:
            tier_rank = 3
            seed_num = 0

    return (gen_num, tier_rank, seed_num, filename)


# Ingest all walk-forward continuous blotters
continuous_blotters = sorted(list(OUTPUT_DIR.glob("blotter_continuous_*.parquet")))
print(f"🔍 Found {len(continuous_blotters)} Continuous Walk-Forward Blotters:")
wf_records = []
for bp in continuous_blotters:
    df_b = pd.read_parquet(bp)
    date_col = "date" if "date" in df_b.columns else "decision_date"
    df_b["date"] = pd.to_datetime(df_b[date_col])
    metrics = calculate_blotter_metrics(df_b)
    metrics["Blotter Name"] = bp.name
    wf_records.append(metrics)

if wf_records:
    df_wf = pd.DataFrame(wf_records)
    print("\n" + "=" * 105)
    print("🏆 CONTINUOUS WALK-FORWARD LEADERBOARD")
    print("=" * 105)

    # Attach hierarchical sorting metadata
    sort_metadata = [parse_blotter_sort_key(name) for name in df_wf["Blotter Name"]]
    df_wf["_gen"] = [m[0] for m in sort_metadata]
    df_wf["_tier"] = [m[1] for m in sort_metadata]
    df_wf["_seed"] = [m[2] for m in sort_metadata]

    # Hierarchical Sort: Gen -> ex_post_blend first -> Seed ID numerically
    df_wf = df_wf.sort_values(
        by=["_gen", "_tier", "_seed", "Blotter Name"]
    ).reset_index(drop=True)

    summary_cols = [
        "Blotter Name",
        "Total Return",
        "BM Return",
        "Excess Return",
        "Sharpe",
        "Sortino",
        "IR",
        "Max DD",
        "Beta",
        "Sessions",
    ]
    display_df = df_wf[summary_cols].copy()
    display_df["Total Return"] = display_df["Total Return"].map(
        lambda x: f"{x*100:+.2f}%"
    )
    display_df["BM Return"] = display_df["BM Return"].map(lambda x: f"{x*100:+.2f}%")
    display_df["Excess Return"] = display_df["Excess Return"].map(
        lambda x: f"{x*100:+.2f}%"
    )
    display_df["Max DD"] = display_df["Max DD"].map(lambda x: f"{x*100:.2f}%")
    display_df["Sharpe"] = display_df["Sharpe"].map(lambda x: f"{x:.3f}")
    display_df["Sortino"] = display_df["Sortino"].map(lambda x: f"{x:.3f}")
    display_df["IR"] = display_df["IR"].map(lambda x: f"{x:.3f}")
    display_df["Beta"] = display_df["Beta"].map(lambda x: f"{x:.2f}")

    # Format across all rows to guarantee bit-level column alignment, then inject group spacers
    lines = display_df.to_string(index=False).splitlines()
    header = lines[0]
    data_lines = lines[1:]

    output_lines = [header]
    prev_gen = None
    for gen, line in zip(df_wf["_gen"], data_lines):
        if prev_gen is not None and gen != prev_gen:
            output_lines.append("")  # Visual demarcation between generation families
        output_lines.append(line)
        prev_gen = gen

    print("\n".join(output_lines))
else:
    print("⚠️ No continuous blotters detected in output directory.")

🔍 Found 43 Continuous Walk-Forward Blotters:

🏆 CONTINUOUS WALK-FORWARD LEADERBOARD
                                  Blotter Name Total Return BM Return Excess Return Sharpe Sortino     IR  Max DD Beta  Sessions
blotter_continuous_ex_post_blend_gen17.parquet     +129.13%   +76.68%       +52.45%  0.862   0.803  0.537 -23.73% 1.27      1108
          blotter_continuous_s42_gen17.parquet     +139.10%   +76.68%       +62.41%  0.839   0.712  0.503 -26.31% 1.27      1108
         blotter_continuous_s101_gen17.parquet      +68.95%   +76.68%        -7.74%  0.743  -0.214 -0.135 -21.53% 1.00      1108
         blotter_continuous_s777_gen17.parquet     +174.51%   +76.68%       +97.82%  0.831   0.847  0.588 -33.51% 1.52      1108
    blotter_continuous_committee_gen17.parquet      +58.30%   +76.68%       -18.38%  0.596  -0.248 -0.184 -25.78% 1.13      1108

blotter_continuous_ex_post_blend_gen18.parquet     +129.56%   +75.04%       +54.52%  0.865   0.828  0.555 -22.75% 1.26      1110
          bl

In [4]:
# ### CELL 4: Constituent & Ex-Post Ensemble Synthesizer & Diversification Audit (Gen 19)

CHUNK_WINDOWS = [
    {"chunk_id": 0, "deploy_start": "2022-03-25", "deploy_end": "2023-03-24"},
    {"chunk_id": 1, "deploy_start": "2023-03-25", "deploy_end": "2024-03-24"},
    {"chunk_id": 2, "deploy_start": "2024-03-25", "deploy_end": "2025-03-24"},
    {"chunk_id": 3, "deploy_start": "2025-03-25", "deploy_end": "2026-09-02"},
]

# SEEDS = [42, 101, 777, 1337, 2026]
SEEDS = [42, 101, 777]
GEN = 23
constituent_blotters: Dict[int, pd.DataFrame] = {}

print("\n" + "=" * 80)
print(f"🔬 GENERATION {GEN} CONSTITUENT AUDIT & DIVERSIFICATION DECOMPOSITION")
print("=" * 80)

# Load constituent seed blotters
for s in SEEDS:
    seed_blotter_path = OUTPUT_DIR / f"blotter_continuous_s{s}_gen{GEN}.parquet"
    if not seed_blotter_path.exists():
        fallback_path = OUTPUT_DIR / f"blotter_continuous_s{s}_gen17.parquet"
        if fallback_path.exists():
            print(
                f"ℹ️ Re-indexing Seed {s} continuous blotter from Gen 17 verified source..."
            )
            df_seed = pd.read_parquet(fallback_path)
            df_seed.to_parquet(seed_blotter_path, index=False)
        else:
            raise FileNotFoundError(
                f"❌ Missing constituent continuous blotter for Seed {s}."
            )
    else:
        df_seed = pd.read_parquet(seed_blotter_path)

    df_seed["date"] = pd.to_datetime(df_seed["date"])
    constituent_blotters[s] = df_seed.sort_values("date").reset_index(drop=True)
    m = calculate_blotter_metrics(df_seed)
    print(
        f"  • Seed {s:>4}: Total Ret = {m['Total Return']*100:>+7.2f}% | Sharpe = {m['Sharpe']:>6.3f} | Beta = {m['Beta']:>4.2f} | Max DD = {m['Max DD']*100:>6.2f}%"
    )

# Load or synthesize Gen Ex-Post Blend
blend_path = OUTPUT_DIR / f"blotter_continuous_ex_post_blend_gen{GEN}.parquet"
dates = constituent_blotters[SEEDS[0]]["date"].to_numpy()
bm_rets = constituent_blotters[SEEDS[0]]["bm_daily_simple_ret"].to_numpy(dtype=float)

ret_matrix = np.column_stack(
    [
        constituent_blotters[s]["net_daily_simple_ret"].to_numpy(dtype=float)
        for s in SEEDS
    ]
)
ex_post_rets = np.mean(ret_matrix, axis=1)

ex_post_equity = np.cumprod(1.0 + ex_post_rets)
bm_equity = np.cumprod(1.0 + bm_rets)
alpha_equity = ex_post_equity / np.maximum(bm_equity, 1e-8)

df_ex_post = pd.DataFrame(
    {
        "date": dates,
        "net_daily_simple_ret": ex_post_rets,
        "bm_daily_simple_ret": bm_rets,
        "alpha_daily_simple_ret": ex_post_rets - bm_rets,
        "agent_equity": ex_post_equity,
        "bm_equity": bm_equity,
        "alpha_equity": alpha_equity,
    }
)
df_ex_post.to_parquet(blend_path, index=False)

# Mathematical Proof: Diversification Ratio & Covariance Decomposition
weights = np.ones(len(SEEDS)) / len(SEEDS)
individual_stds = np.std(ret_matrix, axis=0, ddof=1) * np.sqrt(252)
ensemble_std = np.std(ex_post_rets, ddof=1) * np.sqrt(252)
weighted_vol = float(np.dot(weights, individual_stds))
div_ratio = weighted_vol / (ensemble_std + 1e-8)

corr_matrix = pd.DataFrame(
    np.corrcoef(ret_matrix, rowvar=False),
    index=[f"Seed {s}" for s in SEEDS],
    columns=[f"Seed {s}" for s in SEEDS],
)

ep_metrics = calculate_blotter_metrics(df_ex_post)
constituent_sharpes = [
    calculate_blotter_metrics(constituent_blotters[s])["Sharpe"] for s in SEEDS
]
avg_constituent_sharpe = float(np.mean(constituent_sharpes))

print("\n" + "-" * 80)
print(f"📊 EX-POST ENSEMBLE COVARIANCE & RISK REDUCTION DECOMPOSITION:")
print(f"  • Pairwise Return Correlation Matrix:")
print(corr_matrix.to_string())
print(f"\n  • Constituent Weighted Annualized Volatility : {weighted_vol*100:.2f}%")
print(f"  • Realized Ex-Post Ensemble Annual Volatility: {ensemble_std*100:.2f}%")
print(
    f"  • Absolute Volatility Reduction              : -{(weighted_vol - ensemble_std)*100:.2f}%"
)
print(
    f"  • Diversification Ratio (DR)                 : {div_ratio:.4f}  (Values > 1.0 prove structural diversification)"
)
print(f"  • Constituent Mean Sharpe                    : {avg_constituent_sharpe:.3f}")
print(
    f"  • Ex-Post Blended Sharpe                     : {ep_metrics['Sharpe']:.3f} (Gain: +{ep_metrics['Sharpe'] - avg_constituent_sharpe:+.3f})"
)
# print(f"  • Historical Ex-Ante Committee Sharpe (Gen 17): 0.596 (Dilution Anchor)")
print("-" * 80)


🔬 GENERATION 23 CONSTITUENT AUDIT & DIVERSIFICATION DECOMPOSITION
  • Seed   42: Total Ret =  +23.07% | Sharpe =  0.306 | Beta = 1.24 | Max DD = -25.74%
  • Seed  101: Total Ret =  +79.32% | Sharpe =  0.591 | Beta = 1.28 | Max DD = -27.42%
  • Seed  777: Total Ret = +104.23% | Sharpe =  0.676 | Beta = 1.40 | Max DD = -29.97%

--------------------------------------------------------------------------------
📊 EX-POST ENSEMBLE COVARIANCE & RISK REDUCTION DECOMPOSITION:
  • Pairwise Return Correlation Matrix:
           Seed 42  Seed 101  Seed 777
Seed 42   1.000000  0.888887  0.800174
Seed 101  0.888887  1.000000  0.900046
Seed 777  0.800174  0.900046  1.000000

  • Constituent Weighted Annualized Volatility : 29.33%
  • Realized Ex-Post Ensemble Annual Volatility: 27.97%
  • Absolute Volatility Reduction              : -1.37%
  • Diversification Ratio (DR)                 : 1.0488  (Values > 1.0 prove structural diversification)
  • Constituent Mean Sharpe                    : 0.524
  •

In [7]:
# %% [markdown]
# ### CELL 5: Interactive Visual Master Chart (Dual-Generation Walk-Forward Comparison)

# %%
import re
from typing import Dict, List, Optional, Union
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.paths import OUTPUT_DIR
from core.settings import TradingConfig

# ==============================================================================
# 🎛️ USER CONFIGURATION: Up to 2 generations to compare (e.g. [18, 19] or ['gen18', 'gen19'])
# Default: None -> Automatically detects and selects the last 2 available generations
# ==============================================================================
COMPARE_GENS: Optional[List[Union[int, str]]] = [25, 26]
# COMPARE_GENS: Optional[List[Union[int, str]]] = (
#     None  # e.g., [18, 19] or None for auto-detect
# )
MAX_SEEDS_PER_GEN = 5

# Ensure master chunk calendar demarcations exist
if "CHUNK_WINDOWS" not in globals():
    CHUNK_WINDOWS = [
        {"chunk_id": 0, "deploy_start": "2022-03-25", "deploy_end": "2023-03-24"},
        {"chunk_id": 1, "deploy_start": "2023-03-25", "deploy_end": "2024-03-24"},
        {"chunk_id": 2, "deploy_start": "2024-03-25", "deploy_end": "2025-03-24"},
        {"chunk_id": 3, "deploy_start": "2025-03-25", "deploy_end": "2026-09-02"},
    ]

# Discover available continuous generations
all_blotter_files = list(OUTPUT_DIR.glob("blotter_continuous_*.parquet"))
discovered_gens = sorted(
    list(
        {
            int(m.group(1))
            for f in all_blotter_files
            if (m := re.search(r"gen(\d+)", f.name, re.IGNORECASE))
        }
    )
)

if not discovered_gens:
    raise FileNotFoundError(f"❌ No continuous blotter files found in {OUTPUT_DIR}")

# Resolve target comparison generations
if COMPARE_GENS is None or len(COMPARE_GENS) == 0:
    target_gens = discovered_gens[-2:] if len(discovered_gens) >= 2 else discovered_gens
else:
    parsed_gens = []
    for item in COMPARE_GENS:
        val = (
            int(re.search(r"(\d+)", str(item)).group(1))
            if re.search(r"(\d+)", str(item))
            else None
        )
        if val is not None and val in discovered_gens and val not in parsed_gens:
            parsed_gens.append(val)
    target_gens = parsed_gens[:2] if parsed_gens else discovered_gens[-2:]

print(
    f"🎯 Target Comparison Generations: {target_gens} (Discovered: {discovered_gens})"
)

# Visual Palettes: Gen 0 = Baseline (Cool Blue/Purple), Gen 1 = Challenger (Emerald/Amber)
GEN_THEMES = [
    {
        "blend_color": "#2563EB",  # Royal Blue
        "blend_width": 2.8,
        "blend_dash": "solid",
        "seed_dash": "dot",
        "seed_width": 1.2,
        "palette": ["#93C5FD", "#60A5FA", "#818CF8", "#A78BFA", "#C084FC"],
        "fill_color": "rgba(37, 99, 235, 0.08)",
    },
    {
        "blend_color": "#00CC96",  # Emerald Green
        "blend_width": 3.2,
        "blend_dash": "solid",
        "seed_dash": "dash",
        "seed_width": 1.3,
        "palette": ["#34D399", "#10B981", "#F59E0B", "#FB923C", "#F43F5E"],
        "fill_color": "rgba(0, 204, 150, 0.14)",
    },
]

# Initialize 3-Row Canvas
benchmark_sym = TradingConfig().benchmark
title_str = (
    f"🏆 Walk-Forward Performance & Risk Decomposition: Gen {target_gens[0]} vs Gen {target_gens[1]}"
    if len(target_gens) > 1
    else f"🏆 Walk-Forward Performance & Risk Decomposition: Gen {target_gens[0]}"
)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.45, 0.27, 0.28],
    subplot_titles=(
        f"📈 Cumulative Realized Performance vs Benchmark ({benchmark_sym})",
        "🔥 Strict Geometric Alpha Ratio (V_p / V_bm)",
        "📉 Underwater Drawdown Dynamics",
    ),
)

benchmark_plotted = False

# Iterate over selected generations
for gen_idx, gen in enumerate(target_gens):
    theme = GEN_THEMES[gen_idx % len(GEN_THEMES)]

    # 1. Discover Constituent Seeds (max 5)
    seed_files = list(OUTPUT_DIR.glob(f"blotter_continuous_s*_gen{gen}.parquet"))
    seed_map: Dict[int, pd.DataFrame] = {}
    for sf in seed_files:
        m = re.search(r"_s(\d+)(?:_|\.|$)", sf.name)
        if m:
            s_id = int(m.group(1))
            df_s = pd.read_parquet(sf)
            df_s["date"] = pd.to_datetime(
                df_s["date"] if "date" in df_s.columns else df_s["decision_date"]
            )
            seed_map[s_id] = df_s.sort_values("date").reset_index(drop=True)

    active_seeds = sorted(seed_map.keys())[:MAX_SEEDS_PER_GEN]

    # Plot Benchmark Once from the earliest valid series
    if not benchmark_plotted and active_seeds:
        ref_df = seed_map[active_seeds[0]]
        dates = ref_df["date"]
        bm_curve = ref_df["bm_equity"].to_numpy(dtype=float)
        bm_peaks = np.maximum.accumulate(bm_curve)
        bm_dds = (bm_curve - bm_peaks) / np.maximum(bm_peaks, 1e-8)

        fig.add_trace(
            go.Scatter(
                x=dates,
                y=bm_curve,
                mode="lines",
                name=f"Benchmark ({benchmark_sym})",
                line=dict(color="#111827", width=2.0, dash="dash"),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=np.ones(len(dates)),
                mode="lines",
                name="Benchmark Parity (1.0)",
                line=dict(color="#111827", width=1.0, dash="dot"),
                showlegend=False,
            ),
            row=2,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=bm_dds,
                mode="lines",
                name="Benchmark DD",
                line=dict(color="#6B7280", width=1.2, dash="dash"),
            ),
            row=3,
            col=1,
        )
        benchmark_plotted = True

    # 2. Plot Constituent Seeds
    for s_idx, s in enumerate(active_seeds):
        df_s = seed_map[s]
        m = calculate_blotter_metrics(df_s)
        s_eq = df_s["agent_equity"].to_numpy(dtype=float)
        s_alpha = df_s["alpha_equity"].to_numpy(dtype=float)
        s_peaks = np.maximum.accumulate(s_eq)
        s_dds = (s_eq - s_peaks) / np.maximum(s_peaks, 1e-8)
        color = theme["palette"][s_idx % len(theme["palette"])]

        fig.add_trace(
            go.Scatter(
                x=df_s["date"],
                y=s_eq,
                mode="lines",
                name=f"Gen {gen} s{s} (Sh: {m['Sharpe']:.2f})",
                line=dict(
                    color=color, width=theme["seed_width"], dash=theme["seed_dash"]
                ),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df_s["date"],
                y=s_alpha,
                mode="lines",
                line=dict(
                    color=color, width=theme["seed_width"], dash=theme["seed_dash"]
                ),
                showlegend=False,
            ),
            row=2,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df_s["date"],
                y=s_dds,
                mode="lines",
                line=dict(color=color, width=0.8, dash=theme["seed_dash"]),
                showlegend=False,
            ),
            row=3,
            col=1,
        )

    # 3. Plot Ex-Post Blend Blotter
    blend_file = OUTPUT_DIR / f"blotter_continuous_ex_post_blend_gen{gen}.parquet"
    if blend_file.exists():
        df_b = pd.read_parquet(blend_file)
        df_b["date"] = pd.to_datetime(
            df_b["date"] if "date" in df_b.columns else df_b["decision_date"]
        )
        df_b = df_b.sort_values("date").reset_index(drop=True)
        mb = calculate_blotter_metrics(df_b)

        b_eq = df_b["agent_equity"].to_numpy(dtype=float)
        b_alpha = df_b["alpha_equity"].to_numpy(dtype=float)
        b_peaks = np.maximum.accumulate(b_eq)
        b_dds = (b_eq - b_peaks) / np.maximum(b_peaks, 1e-8)

        is_challenger = gen_idx == (len(target_gens) - 1)
        fig.add_trace(
            go.Scatter(
                x=df_b["date"],
                y=b_eq,
                mode="lines",
                name=f"⭐ Gen {gen} Blend (Sh: {mb['Sharpe']:.2f} | Excess: {mb['Excess Return']*100:+.1f}%)",
                line=dict(
                    color=theme["blend_color"],
                    width=theme["blend_width"],
                    dash=theme["blend_dash"],
                ),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df_b["date"],
                y=b_alpha,
                mode="lines",
                line=dict(
                    color=theme["blend_color"],
                    width=theme["blend_width"] - 0.5,
                    dash=theme["blend_dash"],
                ),
                showlegend=False,
            ),
            row=2,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df_b["date"],
                y=b_dds,
                mode="lines",
                fill="tozeroy" if is_challenger else None,
                fillcolor=theme["fill_color"] if is_challenger else None,
                name=f"Gen {gen} Blend DD",
                line=dict(
                    color=theme["blend_color"], width=1.8, dash=theme["blend_dash"]
                ),
            ),
            row=3,
            col=1,
        )

# Chunk Demarcation Boundaries
for win in CHUNK_WINDOWS[1:]:
    d_val = win["deploy_start"]
    fig.add_vline(x=d_val, line_dash="dash", line_color="#9CA3AF", opacity=0.7)
    fig.add_annotation(
        x=d_val,
        y=0.98,
        xref="x",
        yref="paper",
        text=f"Chunk {win['chunk_id']}",
        showarrow=False,
        xanchor="left",
        font=dict(size=10, color="#6B7280"),
    )

fig.update_layout(
    height=1100,
    width=1280,
    hovermode="x unified",
    margin=dict(t=150, b=50, l=70, r=50),
    title=dict(
        text=title_str,
        y=0.97,
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(size=16, color="#111827"),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=10),
        bgcolor="rgba(255, 255, 255, 0.92)",
        bordercolor="rgba(0, 0, 0, 0.12)",
        borderwidth=1,
    ),
)
fig.update_yaxes(title_text="Wealth Multiplier", row=1, col=1)
fig.update_yaxes(title_text="Alpha (V_p / V_bm)", row=2, col=1)
fig.update_yaxes(title_text="Drawdown %", tickformat=".0%", row=3, col=1)
fig.show()

🎯 Target Comparison Generations: [25, 26] (Discovered: [17, 18, 19, 20, 21, 22, 23, 24, 25, 26])


In [8]:
# CELL 6: Track 2 — Batch Evaluator for Standalone Grid-Sweep Checkpoints
master_csv_path = rl_root / "Grid_sweep.csv"

all_pts = sorted(
    [p for p in checkpoint_dir.rglob("*.pt") if "walk_forward" not in p.parts]
)
print(f"\n🔍 Found {len(all_pts)} standalone checkpoints for Track 2 evaluation.")
print(
    f"👉 Master Champion Blotter Active: blotter_continuous_ex_post_blend_gen{GEN}.parquet"
)
print(f"👉 Proceed to Notebook 03b for Deep Regime Attribution & Factor Diagnostics.")


🔍 Found 140 standalone checkpoints for Track 2 evaluation.
👉 Master Champion Blotter Active: blotter_continuous_ex_post_blend_gen23.parquet
👉 Proceed to Notebook 03b for Deep Regime Attribution & Factor Diagnostics.
